# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane 2: Refresh / Content Opportunity Scoring** — five plain-words contract answers:

1. **One row means:** one **content page** (one pseudonymized `content_hash_id` under one `client_hash_id`) at a **month-end snapshot** — not a day, not a query, not a whole client.
2. **Table(s):** `fact_content_daily_performance` (partition `month=2026-03`) for search + analytics signals; `dim_content` for page metadata (age, last update).
3. **Time window:** **March 2026** (`2026-03-01` → `2026-03-31`) for features; the decision moment is **2026-03-31**. Label proxy compares impressions in the **second half of March vs the first half** (measured, not hand-written).
4. **Predict / rank:** a **decline-risk score** used to sort a refresh review queue — proxy label `is_declining_label = 1` when second-half impressions are below first-half (same idea as the starter CSV's trend proxy).
5. **Deliberately excluded:** **`imp_trend_pct_mar`** (and any half-of-March impression split used to define the label) — it is label-derived, so it must never be a feature (the trap in section 3).

In [1]:
%pip install -q duckdb huggingface_hub python-dotenv scikit-learn pandas

import os
import getpass

from dotenv import load_dotenv

load_dotenv("../../.env")  # repo root — never commit tokens
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_token")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (hf_...): ")

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
SNAP_DATE = "DATE '2026-03-31'"


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields |
|---|---|
| **Feature** | `content_age_days`, `days_since_last_update`, `gsc_impressions_mar`, `avg_position_mar`, `engagement_rate_mar` |
| **Label / proxy** | `is_declining_label` (1 when second-half March impressions < first-half); built from `imp_first_half`, `imp_second_half` |
| **Context** | `client_hash_id`, `content_hash_id` — for grouping, joins, and client-holdout splits only |
| **Excluded** | `imp_trend_pct_mar` — defines the label, so using it as a feature is leakage; GA4 numeric columns when `ga4_data_available IS NOT TRUE` — zeros are fill, not "no engagement" |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain (one row = report_date × client × content)

In [2]:
grain = con.sql(
    f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
    """
).df()
print(f"Duplicate daily keys (expect 0): {len(grain)}")
grain

Duplicate daily keys (expect 0): 0


,client_hash_id,content_hash_id,report_date,c


### Query 2 — Row count and date span (March 2026 slice)

In [3]:
counts = con.sql(
    f"""
    SELECT COUNT(*) AS daily_rows,
           COUNT(DISTINCT content_hash_id) AS unique_pages,
           COUNT(DISTINCT client_hash_id) AS unique_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MAR}
    """
).df()
counts

,daily_rows,unique_pages,unique_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Query 3 — Availability (`ga4_data_available IS TRUE`)

In [4]:
avail = con.sql(
    f"""
    SELECT COUNT(*) AS rows_before,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_after_ga4_filter,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_kept
    FROM {FACT_MAR}
    """
).df()
avail

,rows_before,rows_after_ga4_filter,pct_kept
0,9841378,413966,4.2


### Five features (March 2026, page-level)

Daily rows are rolled up to **one row per page** with `ga4_data_available IS TRUE`, `gsc_data_available IS TRUE`, and at least 100 March impressions. Each feature gets an **available-when** line below the frame.

In [5]:
features = con.sql(
    f"""
    WITH daily AS (
        SELECT *
        FROM {FACT_MAR}
        WHERE ga4_data_available IS TRUE
          AND gsc_data_available IS TRUE
    ),
    page AS (
        SELECT
            d.client_hash_id,
            d.content_hash_id,
            SUM(d.gsc_impressions) AS gsc_impressions_mar,
            AVG(CASE WHEN d.gsc_avg_position > 0 THEN d.gsc_avg_position END) AS avg_position_mar,
            SUM(d.ga4_sessions) AS sessions_mar,
            SUM(d.ga4_engaged_sessions) AS engaged_sessions_mar,
            SUM(CASE WHEN d.report_date <= DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN d.report_date >  DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_second_half
        FROM daily d
        GROUP BY 1, 2
        HAVING SUM(d.gsc_impressions) >= 100
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        DATE_DIFF('day', dc.content_created_date, {SNAP_DATE}) AS content_age_days,
        DATE_DIFF('day', dc.content_updated_date, {SNAP_DATE}) AS days_since_last_update,
        p.gsc_impressions_mar,
        p.avg_position_mar,
        CASE WHEN p.sessions_mar > 0
             THEN 100.0 * p.engaged_sessions_mar / p.sessions_mar END AS engagement_rate_mar,
        CASE WHEN p.imp_second_half < p.imp_first_half THEN 1 ELSE 0 END AS is_declining_label,
        CASE WHEN p.imp_first_half > 0
             THEN 100.0 * (p.imp_second_half - p.imp_first_half) / p.imp_first_half END AS imp_trend_pct_mar
    FROM page p
    JOIN {DIM} dc USING (content_hash_id)
    """
).df()

print(f"Page-level rows: {len(features):,}")
print(f"Observed P(is_declining_label=1): {features['is_declining_label'].mean():.3f}")
print()
for line in [
    "content_age_days — knowable at the decision moment because content_created_date is fixed metadata through 2026-03-31.",
    "days_since_last_update — knowable at the decision moment because content_updated_date is known before we score the queue.",
    "gsc_impressions_mar — knowable at the decision moment because it sums GSC impressions only through 2026-03-31.",
    "avg_position_mar — knowable at the decision moment because it averages March GSC positions reported on or before 2026-03-31.",
    "engagement_rate_mar — knowable at the decision moment because engaged_sessions and sessions are measured only on GA4-available days in March.",
]:
    print("-", line)

features[
    [
        "content_age_days",
        "days_since_last_update",
        "gsc_impressions_mar",
        "avg_position_mar",
        "engagement_rate_mar",
        "is_declining_label",
    ]
].head(8)

Page-level rows: 32,596
Observed P(is_declining_label=1): 0.268

- content_age_days — knowable at the decision moment because content_created_date is fixed metadata through 2026-03-31.
- days_since_last_update — knowable at the decision moment because content_updated_date is known before we score the queue.
- gsc_impressions_mar — knowable at the decision moment because it sums GSC impressions only through 2026-03-31.
- avg_position_mar — knowable at the decision moment because it averages March GSC positions reported on or before 2026-03-31.
- engagement_rate_mar — knowable at the decision moment because engaged_sessions and sessions are measured only on GA4-available days in March.


,content_age_days,days_since_last_update,gsc_impressions_mar,avg_position_mar,engagement_rate_mar,is_declining_label
0,155,-50,3101.0,5.275676,9.375000,0
1,154,-50,2465.0,13.321890,0.000000,1
2,154,-50,13725.0,6.434917,0.000000,0
3,154,-50,9624.0,3.897460,0.000000,1
4,154,-50,90756.0,22.929623,0.851789,1
5,154,-50,4542.0,10.139306,5.555556,1
6,154,-50,6629.0,11.592339,1.162791,1
7,154,-50,13304.0,4.566310,0.000000,1


### The trap — add a label-derived column on purpose, then remove it

`imp_trend_pct_mar` is computed from the same two March halves that define `is_declining_label`. Adding it as a feature should inflate ROC-AUC; we keep the honest score without it.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HONEST_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "gsc_impressions_mar",
    "avg_position_mar",
    "engagement_rate_mar",
]


def quick_auc(df, cols):
    X = df[cols].astype(float)
    X = X.fillna(X.median(numeric_only=True))
    y = df["is_declining_label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    )
    pipe.fit(X_train, y_train)
    return roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])


auc_honest = quick_auc(features, HONEST_FEATURES)
auc_leaky = quick_auc(features, HONEST_FEATURES + ["imp_trend_pct_mar"])

print(f"Honest ROC-AUC (5 features only): {auc_honest:.3f}")
print(f"With label-derived imp_trend_pct_mar added: {auc_leaky:.3f}")
print("Keeping the honest number — imp_trend_pct_mar stays excluded.")

Honest ROC-AUC (5 features only): 0.653
With label-derived imp_trend_pct_mar added: 0.997
Keeping the honest number — imp_trend_pct_mar stays excluded.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation — GA4 coverage is uneven in March 2026.**

Only about **4%** of daily fact rows in this partition pass `ga4_data_available IS TRUE` (see Query 3). That means engagement-based ranking signals are **directional decision-support for GA4-connected pages only**, not a complete picture of every client's inventory. Early client history is often GSC-only (`ga4_data_available = FALSE` with zero-filled GA4 columns), and per-client history depth still differs — so a single global calendar window can overstate how much of the panel is truly comparable.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.